# Day 17 · Light & dark — The 3 a.m. City

**Question.** The address grid promises a bus on every half-mile arterial. How much of that grid is
still running at 3 a.m., and who is left in the dark?

**Approach.** Every CTA bus leaving every stop on an ordinary weekday, counted per hour in quarter-mile
blocks of the address grid. On the plate each block is lit by its departures (brighter = more buses)
and stands up toward a light in the northwest, so busy blocks cast a short shadow. The plate shows
3 a.m.; the day page steps through all 24 hours. Residents (2020 Census) and night-shift commuters
(ACS: leave home between midnight and 5 a.m.) are measured against the half mile to a stop with a bus
in the hour.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | CTA GTFS on disk, city limits, 2020 Census blocks, ACS B08302 |
| 3. Validate | Service day, bus routes, stops |
| 4. Transform | Departures per stop and hour; quarter-mile blocks |
| 5. Analyze | Lit blocks, all-night routes, residents and night commuters within a half mile |
| 6. QA | Another weekday; block sums |
| 7. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import zipfile
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from shapely import contains_xy
from shapely.geometry import Polygon, box
from shapely.ops import unary_union

from atlas.grid import miles_from_origin
from atlas.ledger import Ledger, write_plate

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger('day17')
DAY = Path('.')
REPO = DAY.resolve().parents[1]
OUT = DAY / 'output'
OUT.mkdir(parents=True, exist_ok=True)

GTFS_ZIP = REPO / 'reference' / 'raw' / 'gtfs' / 'cta_gtfs_2026-11-08.zip'
GTFS_URL = 'https://www.transitchicago.com/developers/gtfs/'
SERVICE_DAY = date(2026, 11, 17)          # an ordinary Tuesday
QA_ZIP = REPO / 'reference' / 'raw' / 'gtfs' / 'cta_gtfs_2026-11-01.zip'   # the week before's snapshot
Q = 0.25                                  # block, miles
X0, X1, Y0, Y1 = -16.0, 6.0, -17.5, 13.0  # block centers span the city (miles from State & Madison)
HALF = 0.5                                # the half mile, straight line
NIGHT = 3                                 # the plate's hour
ACS = REPO / 'reference' / 'raw' / 'census' / 'acsdt5y2024-b08302.dat'
ACS_URL = 'https://www2.census.gov/programs-surveys/acs/summary_file/2024/table-based-SF/data/5YRData/acsdt5y2024-b08302.dat'
CENSUS_URL = 'https://www2.census.gov/programs-surveys/decennial/2020/data/01-Redistricting_File--PL_94-171/Illinois/il2020.pl.zip'
# Chicago's nine sides, by community area (as Plate X): the grouping of Wikipedia's 'Community areas in Chicago' (unofficial)
SIDES = {
    'Central': [8, 32, 33],
    'North Side': [5, 6, 7, 21, 22],
    'Far North Side': [1, 2, 3, 4, 9, 10, 11, 12, 13, 14, 76, 77],
    'Northwest Side': [15, 16, 17, 18, 19, 20],
    'West Side': [23, 24, 25, 26, 27, 28, 29, 30, 31],
    'South Side': [34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 60, 68, 69],
    'Southwest Side': [56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67],
    'Far Southwest Side': [70, 71, 72, 73, 74, 75],
    'Far Southeast Side': [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55],
}
SIDE_OF = {ca: s for s, cas in SIDES.items() for ca in cas}
assert sorted(SIDE_OF) == list(range(1, 78))

## 2. Pull
Everything is on disk: the CTA GTFS snapshot used since Day 13, the city limits, the 2020 Census blocks (`atlas.census`) and the ACS table B08302 (time leaving home to go to work), a keyless table-based summary file.

In [ ]:
Z = zipfile.ZipFile(GTFS_ZIP)


def rd(name, z=Z, **kw):
    df = pd.read_csv(z.open(name), dtype=str, skipinitialspace=True, **kw)
    df.columns = df.columns.str.strip()
    return df


GTFS_DATE = GTFS_ZIP.stem.split('_')[-1]
boundary = json.load(open(REPO / 'reference' / 'city-boundary.json'))
blocks = pd.read_csv(REPO / 'reference' / 'chicago-blocks-2020.csv', dtype={'geoid': str})
if not ACS.exists():
    import requests
    ACS.write_bytes(requests.get(ACS_URL, timeout=300).content)
acs = pd.read_csv(ACS, sep='|', dtype=str, usecols=['GEO_ID', 'B08302_E001', 'B08302_E002'])
acs = acs[acs.GEO_ID.str.startswith('1400000US17031')].assign(tract=lambda d: d.GEO_ID.str[9:])
acs[['B08302_E001', 'B08302_E002']] = acs[['B08302_E001', 'B08302_E002']].astype(float)
log.info('GTFS %s; blocks %d, residents %d; Cook County tracts %d', GTFS_DATE, len(blocks), blocks['pop'].sum(), len(acs))

## 3. Validate
The service day and its bus trips. A stop time past 24:00 belongs to the next morning; folded onto the clock, one service day gives every hour of an ordinary weekday once.

In [ ]:
def services(day, z=Z):
    cal = rd('calendar.txt', z)
    stamp, wd = day.strftime('%Y%m%d'), day.strftime('%A').lower()
    sv = set(cal[(cal.start_date <= stamp) & (cal.end_date >= stamp) & (cal[wd] == '1')].service_id)
    ex = rd('calendar_dates.txt', z)
    ex = ex[ex.date == stamp]
    return (sv | set(ex[ex.exception_type == '1'].service_id)) - set(ex[ex.exception_type == '2'].service_id)


routes = rd('routes.txt')
BUS = set(routes.loc[routes.route_type == '3', 'route_id'])
trips = rd('trips.txt', usecols=['route_id', 'service_id', 'trip_id'])
stops = rd('stops.txt', usecols=['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'location_type'])
stops = stops[stops.location_type.fillna('0') == '0'].copy()
stops['x'], stops['y'] = miles_from_origin(stops.stop_lat.astype(float), stops.stop_lon.astype(float))
log.info('bus routes %d, rail routes %d', len(BUS), (routes.route_type == '1').sum())
assert len(BUS) > 100


def departures(day, z=Z):
    """Departures per (route, stop, clock hour) on one service day, buses and trains; bus = True for buses."""
    tr = rd('trips.txt', z, usecols=['route_id', 'service_id', 'trip_id'])
    st = rd('stops.txt', z, usecols=['stop_id', 'stop_lat', 'stop_lon'])
    st['x'], st['y'] = miles_from_origin(st.stop_lat.astype(float), st.stop_lon.astype(float))
    t = tr[tr.service_id.isin(services(day, z))]
    parts = []
    for ch in pd.read_csv(z.open('stop_times.txt'), dtype=str, skipinitialspace=True, chunksize=2_000_000,
                          usecols=['trip_id', 'departure_time', 'stop_id', 'pickup_type']):
        ch.columns = ch.columns.str.strip()
        ch = ch.dropna(subset=['departure_time'])
        ch = ch[ch.pickup_type.fillna('0').str.strip() != '1'].merge(t, on='trip_id')   # boarding allowed
        ch['h'] = ch.departure_time.str.strip().str.split(':').str[0].astype(int) % 24
        parts.append(ch.groupby(['route_id', 'stop_id', 'h']).size().rename('n').reset_index())
    d = pd.concat(parts).groupby(['route_id', 'stop_id', 'h']).n.sum().reset_index()
    d['bus'] = d.route_id.isin(BUS)
    log.info('%s: %d trips, %d bus departures, %d train departures', day, len(t), d.n[d.bus].sum(), d.n[~d.bus].sum())
    return d.merge(st[['stop_id', 'x', 'y']], on='stop_id')


every = departures(SERVICE_DAY)          # buses and trains: the half mile for residents
dep = every[every.bus].copy()            # buses: the blocks
assert dep.n.sum() > 500_000

## 4. Transform
Quarter-mile blocks set so the arterials run through the middle of a block, never along its edge. The north-south arterials sit on the quarter-mile lines from State Street. The east-west arterials, measured here, sit about a sixteenth of a mile south of their lines from Madison (Madison Street’s own stops among them), so the rows are shifted south by 1/16 mile. A block belongs to the city when it overlaps the city limits; it counts the buses leaving stops inside the city within it.

In [ ]:
# city limits in miles, from the raw portal snapshot (all parts, holes kept)
gj = json.load(open(REPO / 'reference' / 'raw' / 'city_boundary_2026-11-01.geojson'))
polys = []
for f in gj['features']:
    g = f['geometry']
    for p in (g['coordinates'] if g['type'] == 'MultiPolygon' else [g['coordinates']]):
        rings = [np.column_stack(miles_from_origin(np.array(r)[:, 1], np.array(r)[:, 0])) for r in p]
        polys.append(Polygon(rings[0], rings[1:]))
city = unary_union(polys)
log.info('city: %.1f sq mi', city.area)

# where the east-west arterials' stops sit, relative to their half-mile lines
EW = {'20': 0, '66': 1, '72': 2, '77': 4, '55': -6, '63': -7, '79': -9, '87': -10}   # route: its line, miles (Madison, Chicago, North, Belmont, Garfield, 63rd, 79th, 87th)
off = {r: float(dep[dep.route_id == r].drop_duplicates('stop_id').y.median() - m) for r, m in EW.items()}
log.info('east-west arterial stops, median miles off their line: %s', {r: round(v, 3) for r, v in off.items()})
SHIFT = -1 / 16
assert all(abs(v - SHIFT) < 0.04 for v in off.values())

nx = int(round((X1 - X0) / Q)) + 1
cx = X0 + Q * np.arange(nx)
cy = Y0 + SHIFT + Q * np.arange(int(round((Y1 - Y0) / Q)) + 1)
lo, hi = cy - Q / 2, cy + Q / 2
ny = len(cy)
inside = np.zeros((ny, nx), bool)
for j in range(ny):
    for i in range(nx):
        inside[j, i] = city.intersects(box(cx[i] - Q / 2, lo[j], cx[i] + Q / 2, hi[j]))
log.info('city blocks %d; rows %d', inside.sum(), ny)

dep['in_city'] = contains_xy(city, dep.x.to_numpy(), dep.y.to_numpy())
dc = dep[dep.in_city].copy()
dc['i'] = np.round((dc.x - X0) / Q).astype(int)
dc['j'] = np.searchsorted(lo, dc.y.to_numpy()) - 1   # rows south to north
assert inside[dc.j, dc.i].all(), 'a city stop fell in a block outside the city'
A = np.zeros((24, ny, nx), int)
np.add.at(A, (dc.h.to_numpy(), dc.j.to_numpy(), dc.i.to_numpy()), dc.n.to_numpy())
hours = A.sum(axis=(1, 2))
log.info('city bus departures a day %d; by hour %s', hours.sum(), hours.tolist())

## 5. Analyze
Lit blocks per hour; bus routes running all night; residents and night-shift commuters within a half mile (straight line) of a bus stop or 'L' platform with a departure in the hour. The map draws buses only, but the Red and Blue lines run all night, so the half mile counts trains too.

In [ ]:
lit = (A > 0).sum(axis=(1, 2))
peak = int(np.argmax(hours))
byroute = dep.groupby(['route_id', 'h']).n.sum().unstack(fill_value=0).reindex(columns=range(24), fill_value=0)
at3 = sorted(byroute.index[byroute[NIGHT] > 0], key=lambda r: (len(r), r))
short = routes.set_index('route_id').route_short_name


def starts(day, z=Z):
    """Trips starting their run (first stop), per route and clock hour."""
    tr = rd('trips.txt', z, usecols=['route_id', 'service_id', 'trip_id'])
    tr = tr[tr.service_id.isin(services(day, z))]
    parts = []
    for ch in pd.read_csv(z.open('stop_times.txt'), dtype=str, skipinitialspace=True, chunksize=2_000_000,
                          usecols=['trip_id', 'departure_time', 'stop_sequence']):
        ch.columns = ch.columns.str.strip()
        parts.append(ch[ch.trip_id.isin(set(tr.trip_id))])
    st = pd.concat(parts).assign(seq=lambda d: d.stop_sequence.astype(int))
    first = st.loc[st.groupby('trip_id').seq.idxmin()].merge(tr, on='trip_id')
    first['h'] = first.departure_time.str.strip().str.split(':').str[0].astype(int) % 24
    return first.groupby(['route_id', 'h']).size().unstack(fill_value=0).reindex(columns=range(24), fill_value=0)


# all night: a trip leaves the end of the line in every hour from 1 to 4 a.m. (late runs finishing after
# 1 a.m. and early runs starting before 4 a.m. do not make a route all-night)
S = starts(SERVICE_DAY)
night_routes = S.index[(S[[1, 2, 3, 4]] > 0).all(axis=1)]
allnight = sorted([r for r in night_routes if r in BUS], key=lambda r: (len(r), r))
railAllNight = sorted([r for r in night_routes if r not in BUS])
medLit = [float(np.median(A[h][A[h] > 0])) for h in range(24)]
log.info('rail lines all night: %s; median buses in a lit block: peak %.0f, 3 a.m. %.0f', railAllNight, medLit[peak], medLit[NIGHT])
log.info('peak %d:00 (%d), 3 a.m. %d; lit blocks peak %d, 3 a.m. %d', peak, hours[peak], hours[NIGHT], lit[peak], lit[NIGHT])
log.info('bus routes all night: %d %s; with a bus at 3 a.m.: %d', len(allnight), allnight, len(at3))

# residents: 2020 block internal points; night commuters: ACS tract counts allocated by population
bx, by = miles_from_origin(blocks['lat'], blocks['lon'])
pop = blocks['pop'].to_numpy(float)
b = blocks.assign(tract=blocks.geoid.str[:11]).merge(acs, on='tract', how='left')
share = b['pop'] / b.groupby('tract')['pop'].transform('sum')
night = (b.B08302_E002 * share).fillna(0).to_numpy()
workers = (b.B08302_E001 * share).fillna(0).to_numpy()
log.info('commuters %.0f; leave home 12:00-4:59 a.m. %.0f (%.1f%%)', workers.sum(), night.sum(), 100 * night.sum() / workers.sum())

# every CTA bus stop and 'L' platform counts for the half mile, inside the city or not
near = np.zeros((24, len(blocks)), bool)
nearBus = np.zeros(len(blocks), bool)
for h in range(24):
    s = every[(every.h == h)].drop_duplicates('stop_id')
    d, _ = cKDTree(s[['x', 'y']].to_numpy()).query(np.column_stack([bx, by]), distance_upper_bound=HALF)
    near[h] = np.isfinite(d)
s = dep[dep.h == NIGHT].drop_duplicates('stop_id')
nearBus = np.isfinite(cKDTree(s[['x', 'y']].to_numpy()).query(np.column_stack([bx, by]), distance_upper_bound=HALF)[0])
within = (near * pop).sum(axis=1) / pop.sum()
dark = {'residents': float(pop[~near[NIGHT]].sum()), 'share': float(1 - within[NIGHT]),
        'night': float(night[~near[NIGHT]].sum()), 'nightShare': float(night[~near[NIGHT]].sum() / night.sum()),
        'peakShare': float(1 - within[peak]), 'busOnly': float(pop[~nearBus].sum() / pop.sum())}
log.info('beyond a half mile of a running bus: 3 a.m. %.1f%% (%d), peak %.1f%%; night commuters %.1f%% (%d)',
         100 * dark['share'], dark['residents'], 100 * dark['peakShare'], 100 * dark['nightShare'], dark['night'])

side = blocks.commarea.map(SIDE_OF)
bySide = (pd.DataFrame({'side': side, 'pop': pop, 'dark': pop * ~near[NIGHT]})
          .groupby('side').sum().assign(share=lambda d: d.dark / d['pop']).sort_values('share', ascending=False))
print(bySide.round(3))

## 6. QA
The week before's GTFS snapshot should give the same night; the blocks hold every bus departure at city stops.

In [ ]:
qa_all = departures(SERVICE_DAY, zipfile.ZipFile(QA_ZIP))
qa_night = int(qa_all[qa_all.bus & (qa_all.h == NIGHT)].n.sum())
all_night = int(dep[dep.h == NIGHT].n.sum())
log.info('3 a.m. bus departures, all stops: snapshot %s %d, snapshot %s %d', GTFS_DATE, all_night, QA_ZIP.stem[-10:], qa_night)
assert abs(qa_night - all_night) / all_night < 0.05
assert A.sum() == dc.n.sum()
assert hours[NIGHT] < hours[peak] / 5
assert lit[NIGHT] < lit[peak]

## 7. Export
Blocks with a bus in any hour, with their 24 counts; routes at 3 a.m. per block; hourly totals; findings text.

In [ ]:
names = {}
for (j, i), g in dc[dc.h == NIGHT].groupby(['j', 'i']):
    names[(j, i)] = sorted(set(short[g.route_id]), key=lambda r: (len(r), r))
ever = np.argwhere(inside & (A.sum(axis=0) > 0))
cells = [[int(i), int(j), A[:, j, i].tolist(), names.get((j, i), [])] for j, i in ever]
dark_cells = [[int(i), int(j)] for j, i in np.argwhere(inside & (A.sum(axis=0) == 0))]

# the inset: the Far Southwest Side on block edges (Cicero to Halsted, 79th to 119th Streets); its
# arterials named where their own bus stops sit (the grid drifts: Ashland runs 1.8 miles west of State
# here, not 2), snapped to the block centers
INSET = dict(west=-6.125, east=-0.375, south=-14.1875, north=-8.9375)
# (no bus runs on Western south of 79th: it sits midway between Kedzie and Ashland, as its address)
INSET_ROUTES = {'x': [('53A', 'PULASKI'), ('52A', 'KEDZIE'), (None, 'WESTERN'), ('9', 'ASHLAND'), ('8A', 'HALSTED')],
                'y': [('79', '79TH'), ('87', '87TH'), ('95', '95TH'), ('103', '103RD'), ('112', '111TH')]}
inbox = dep[(dep.x > INSET['west']) & (dep.x < INSET['east']) & (dep.y > INSET['south']) & (dep.y < INSET['north'])]
streets = {}
for ax, pairs in INSET_ROUTES.items():
    centers = cx if ax == 'x' else cy
    at = {}
    for r, name in pairs:
        if r is None:
            continue
        v = inbox[inbox.route_id == r].drop_duplicates('stop_id')[ax]
        assert len(v) >= 3, r
        at[name] = float(v.median())
    if 'WESTERN' in dict((n, 1) for _, n in pairs):
        at['WESTERN'] = (at['KEDZIE'] + at['ASHLAND']) / 2
    streets[ax] = [[round(float(centers[np.argmin(np.abs(centers - at[n]))]), 4), n] for _, n in pairs]
log.info('inset streets %s', streets)

# the sky: the sun's altitude in Chicago at each hour's midpoint on the service day (NOAA's simple
# formulas; Central Standard Time), as a 0-1 blend from night (below -6°, civil twilight) to day (above +6°)
def sun_altitude(day, hour, lat=41.88, lon=-87.63, utc_offset=-6):
    n = day.timetuple().tm_yday
    g = 2 * np.pi / 365 * (n - 1 + (hour - 12) / 24)
    decl = 0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g) + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g)
    eqt = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g) - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    solar = hour * 60 + eqt + 4 * lon - 60 * utc_offset
    ha = np.radians(solar / 4 - 180)
    la = np.radians(lat)
    return np.degrees(np.arcsin(np.sin(la) * np.sin(decl) + np.cos(la) * np.cos(decl) * np.cos(ha)))


alt = np.array([sun_altitude(SERVICE_DAY, h + 0.5) for h in range(24)])
sky = np.clip((alt + 6) / 12, 0, 1)
log.info('sun altitude at the half hours: %s', np.round(alt, 1).tolist())
assert sky[NIGHT] == 0 and sky[12] == 1

fmt = lambda v: f'{v:,.0f}'
rnd = lambda v, n=-3: fmt(round(v, n))
worst = bySide.index[0]
finding = (f'At 3 a.m., {lit[NIGHT]:,} of the {lit[peak]:,} quarter-mile blocks with a bus at the morning peak still have one. '
           f'About {rnd(dark["residents"], -4)} Chicagoans live more than a half mile from a bus or train leaving that hour.')
plate = dict(
    gtfsDate=GTFS_DATE, serviceDay=SERVICE_DAY.isoformat(), night=NIGHT, peak=peak,
    grid=dict(x0=X0, step=Q, nx=nx, rows=[[round(float(c), 4), round(float(a), 4), round(float(b), 4)] for c, a, b in zip(cy, lo, hi)]),
    cells=cells, dark=dark_cells, inset=dict(box=INSET, streets=streets),
    hours=dict(departures=hours.tolist(), lit=lit.tolist(), within=[round(float(v), 4) for v in within],
               sunAltitude=[round(float(v), 1) for v in alt], sky=[round(float(v), 3) for v in sky]),
    routes=dict(allNight=[short[r] for r in allnight], at3=[short[r] for r in at3], bus=len(BUS), railAllNight=railAllNight),
    summary=dict(
        departuresDay=int(hours.sum()), cityBlocks=int(inside.sum()), residents=int(pop.sum()),
        commuters=float(workers.sum()), nightCommuters=float(night.sum()), dark=dark,
        sides=[[s, float(r.share), float(r.dark)] for s, r in bySide.iterrows()], worstSide=worst,
        maxBlock=[int(A[peak].max()), int(A[NIGHT].max())], medianLit=medLit,
    ),
    qa=dict(night=all_night, nightPrevSnapshot=qa_night),
    sources=dict(gtfs=GTFS_URL, census=CENSUS_URL, acs='https://data.census.gov/table/ACSDT5Y2024.B08302',
                 sides='https://en.wikipedia.org/wiki/Community_areas_in_Chicago'),
)
ledger = Ledger(source='CTA GTFS', dataset_id='GTFS schedule', url=GTFS_URL, pulled=date.fromisoformat(GTFS_DATE),
                units='bus departures per quarter-mile block per hour',
                method=['every bus leaving every stop on Tuesday 17 Nov 2026, times past 24:00 folded onto the morning',
                        'quarter-mile blocks centered on the quarter-mile lines of the address grid'],
                caveats=['scheduled service, not real-time', 'straight-line half mile'])
write_plate(OUT / 'plate.json', day=17, ledger=ledger, finding=finding, payload=plate)
log.info('%s; %d cells', finding, len(cells))